# Tweety C# / IKVM - Revision de Croyances (Port .NET du notebook Python)

Ce notebook porte en **C# / .NET** (via IKVM, sans JVM) le module **belief-revision** de
[TweetyProject](https://tweetyproject.org/). Il fait partie de l'Epic #4667 (portage .NET de la
serie Tweety) et prolonge les notebooks deja portes : logiques de base, semantique, FOL,
argumentation de Dung et ASPIC+.

Le pendant Python est [`Tweety-4-Belief-Revision.ipynb`](Tweety-4-Belief-Revision.ipynb).

## De quoi parle la revision de croyances ?

Un agent rationnel doit parfois **integrer une information qui contredit ce qu'il croit deja**.
Ajouter naivement la nouvelle formule a sa base la rendrait **incoherente** (elle impliquerait
tout et n'importe quoi). La **theorie AGM** (Alchourron, Gardenfors, Makinson, 1985) formalise
les trois operations du changement de croyance :

- **Expansion** (`K + phi`) : on ajoute `phi` sans se soucier de la coherence.
- **Contraction** (`K - phi`) : on retire juste assez de croyances pour que `K` n'implique plus `phi`.
- **Revision** (`K * phi`) : on integre `phi` **en preservant la coherence**, quitte a abandonner
  d'anciennes croyances.

L'**identite de Levi** relie les trois : `K * phi = (K - !phi) + phi`. On contracte d'abord par la
negation de l'information, puis on expanse. Tweety implemente ces operateurs ; ce notebook les
execute reellement sur des bases propositionnelles.

## Comment Tweety tourne en .NET (sans JVM)

Tweety est une bibliotheque **Java**. On la recompile en assembly **.NET** avec **IKVM 8.15.0** :
le bytecode Java devient du code .NET, appelable directement en C# depuis le kernel `.net-csharp`.
La recette de build (Maven shade + `<IkvmReference>`) est documentee dans
[`dotnet-build/`](dotnet-build/).

### 1. Configuration du runtime IKVM

In [1]:
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "nuget: IKVM.Image.runtime.win-x64, 8.15.0"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages IKVM, 8.15.0 IKVM.Image, 8.15.0

In [2]:
using System.IO;

string ikvmVer = "8.15.0", ikvmRid = "win-x64";
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome    = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);

void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}

if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
AppContext.SetData("IKVM.Home", ikvmHome);

bool tzdbOk = File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat"));
Console.WriteLine($"IKVM 8.15.0 pret (home=ikvm-home-{ikvmVer}-{ikvmRid}, tzdb={tzdbOk})");

IKVM 8.15.0 pret (home=ikvm-home-8.15.0-win-x64, tzdb=True)


### 2. Chargement de la DLL Tweety beliefdynamics

La DLL `org.tweetyproject.tweety-beliefdynamics.dll` est un fat-jar shade (beliefdynamics +
ses dependances : logique propositionnelle, SAT, math) recompile par IKVM. Elle expose les
operateurs de revision, de contraction par noyaux et de revision selective.

In [3]:
#r "org.tweetyproject.tweety-beliefdynamics.dll"

### 3. Le probleme : une base de croyances incoherente

Construisons une base `K = { a, a => b, !c }`, coherente. Puis observons que lui ajouter
naivement l'information `!b` la rend **incoherente** : comme `a` et `a => b` impliquent `b`,
la base contiendrait a la fois `b` (deduit) et `!b`. C'est exactement la situation que la
revision doit resoudre proprement.

In [4]:
using org.tweetyproject.logics.pl.syntax;
using org.tweetyproject.logics.pl.parser;
using org.tweetyproject.logics.pl.reasoner;

var parser = new PlParser();
var reasoner = new SimplePlReasoner();

// Test de coherence sans solveur SAT externe : une base est INCOHERENTE si et seulement
// si elle entraine une contradiction (elle n'a alors aucun modele). On reutilise donc le
// raisonneur propositionnel : query(base, "a && !a") vaut true ssi la base est incoherente.
var faux = (PlFormula) parser.parseFormula("a && !a");

// Base de croyances initiale  K = { a, a=>b, !c }  (coherente).
var K = new PlBeliefSet();
K.add((PlFormula) parser.parseFormula("a"));
K.add((PlFormula) parser.parseFormula("a=>b"));
K.add((PlFormula) parser.parseFormula("!c"));
Console.WriteLine($"Base initiale         K = {K}");
Console.WriteLine($"K incoherente ?           {reasoner.query(K, faux)}");

// Nouvelle information  phi = !b. Or a et a=>b impliquent b : phi contredit K.
// L'union naive  K + [ !b ]  est donc incoherente.
var naive = new PlBeliefSet();
naive.add((PlFormula) parser.parseFormula("a"));
naive.add((PlFormula) parser.parseFormula("a=>b"));
naive.add((PlFormula) parser.parseFormula("!c"));
naive.add((PlFormula) parser.parseFormula("!b"));
Console.WriteLine($"Union naive K + [!b]  = {naive}");
Console.WriteLine($"Union naive incoherente ? {reasoner.query(naive, faux)}");

Base initiale         K = { (a=>b), a, !c }


K incoherente ?           false


Union naive K + [!b]  = { (a=>b), a, !b, !c }


Union naive incoherente ? true


#### Interpretation

L'union naive est incoherente : le raisonneur le confirme — elle entraine la contradiction
`a && !a`, donc la question « incoherente ? » passe de `false` (pour `K`) a `true` (pour l'union).
Une base incoherente implique **toute** formule (principe d'explosion), elle est donc inutilisable
pour raisonner. La revision AGM va integrer `!b` **sans** produire cette explosion.

### 4. Contraction par noyaux (kernel contraction)

Avant de reviser, il faut savoir **contracter**. La *contraction par noyaux* (Hansson) procede
ainsi : un **noyau** de `K` par rapport a `b` est un sous-ensemble **minimal** de `K` qui implique
`b`. Pour que `K` cesse d'impliquer `b`, il faut « couper » au moins une formule dans **chaque**
noyau : c'est le role de la **fonction d'incision**. Ici :

- `SimplePlReasoner` fournit les noyaux (`KernelProvider`),
- `RandomIncisionFunction` choisit la formule a retirer dans chaque noyau.

In [5]:
using java.util;
using org.tweetyproject.logics.pl.reasoner;
using org.tweetyproject.beliefdynamics.kernels;

// La contraction retire juste ce qu'il faut pour que la base n'implique plus b.
// KernelContractionOperator(fonction d'incision, fournisseur de noyaux).
//   - SimplePlReasoner joue le role de KernelProvider (il enumere les noyaux).
//   - RandomIncisionFunction choisit dans chaque noyau la formule a couper.
var baseK = new HashSet();
baseK.add((PlFormula) parser.parseFormula("a"));
baseK.add((PlFormula) parser.parseFormula("a=>b"));
baseK.add((PlFormula) parser.parseFormula("!c"));

var aRetirer = new HashSet();
aRetirer.add((PlFormula) parser.parseFormula("b"));

var contraction = new KernelContractionOperator(new RandomIncisionFunction(), new SimplePlReasoner());
var contractee = contraction.contract(baseK, aRetirer);
Console.WriteLine($"K contractee par b = {contractee}");

// La base contractee ne doit plus entrainer b. On reconstruit une PlBeliefSet a partir
// de la collection retournee, puis on interroge le raisonneur.
var contracteeKb = new PlBeliefSet();
var itc = contractee.iterator();
while (itc.hasNext()) contracteeKb.add((PlFormula) itc.next());
var b = (PlFormula) parser.parseFormula("b");
Console.WriteLine($"Entraine encore b ? {reasoner.query(contracteeKb, b)}");

K contractee par b = [a, !c]


Entraine encore b ? false


#### Interpretation

La base contractee n'implique plus `b` : la contraction a retire l'une des formules
responsables (`a` ou `a => b`). La fonction d'incision etant **aleatoire**, la formule
exactement retiree peut varier d'une execution a l'autre — mais la propriete garantie
(« ne plus impliquer `b` ») est, elle, toujours respectee.

### 5. Revision par l'identite de Levi

On assemble maintenant la revision complete. `LeviMultipleBaseRevisionOperator` prend un
operateur de contraction et un operateur d'expansion, et applique l'identite de Levi
`K * phi = (K - !phi) + phi`. Revisons `K` par `!b`.

In [6]:
using org.tweetyproject.beliefdynamics;

// Identite de Levi :  K * phi  =  (K - !phi) + phi
//   revision = contraction par la negation de l'information, puis expansion.
var expansion = new DefaultMultipleBaseExpansionOperator();
var levi = new LeviMultipleBaseRevisionOperator(contraction, expansion);

var nouvelleInfo = new HashSet();
nouvelleInfo.add((PlFormula) parser.parseFormula("!b"));

var revisee = levi.revise(baseK, nouvelleInfo);
Console.WriteLine($"K revisee par !b (Levi) = {revisee}");

var reviseeKb = new PlBeliefSet();
var itr = revisee.iterator();
while (itr.hasNext()) reviseeKb.add((PlFormula) itr.next());
var notB = (PlFormula) parser.parseFormula("!b");
Console.WriteLine($"Resultat incoherent ?     {reasoner.query(reviseeKb, faux)}");
Console.WriteLine($"Contient bien !b ?        {reviseeKb.contains(notB)}");

K revisee par !b (Levi) = [a, !b, !c]


Resultat incoherent ?     false


Contient bien !b ?        True


#### Interpretation

Le resultat est **coherent** et contient bien la nouvelle information `!b`. La revision a
sacrifie juste ce qu'il fallait des anciennes croyances pour accueillir `!b` sans contradiction :
c'est la difference fondamentale avec l'union naive de la section 3.

### 6. Exemple guide : le pingouin Tweety (raisonnement non-monotone)

La bibliotheque tire son nom de **Tweety**, l'oiseau emblematique du raisonnement non-monotone.
Partons de trois croyances : Tweety est un pingouin (`p`), un pingouin est un oiseau (`p => b`),
un oiseau vole (`b => f`). La base **conclut donc que Tweety vole**. Puis on **observe** que ce
pingouin ne vole pas (`!f`). Pour integrer l'observation, la revision doit **abandonner une regle
par defaut** de la chaine de raisonnement `p => b => f`. C'est un probleme non-trivial : l'incoherence
est **derivee** (via une chaine d'implications), pas directe.

Le notebook affiche la ou les croyances effectivement sacrifiees. Notez qu'une contraction AGM
n'est **pas unique** : plusieurs retraits minimaux restaurent la coherence (ici retirer `p => b`
*ou* `b => f`). Le choix depend de la **fonction d'incision** (`RandomIncisionFunction`) ; un
operateur muni de priorites sur les croyances retirerait preferentiellement la regle la moins fiable.

In [7]:
// Exemple guide : le pingouin Tweety (raisonnement non-monotone).
// p = "est un pingouin", b = "est un oiseau", f = "vole".
// K = { p, p=>b, b=>f }  =>  la base conclut que Tweety vole.
var Kt = new HashSet();
Kt.add((PlFormula) parser.parseFormula("p"));
Kt.add((PlFormula) parser.parseFormula("p=>b"));
Kt.add((PlFormula) parser.parseFormula("b=>f"));

var KtKb = new PlBeliefSet();
var itk = Kt.iterator();
while (itk.hasNext()) KtKb.add((PlFormula) itk.next());
var f = (PlFormula) parser.parseFormula("f");
Console.WriteLine($"K conclut que Tweety vole (f) ? {reasoner.query(KtKb, f)}");

// Observation : ce pingouin ne vole PAS. On revise K par !f.
var obs = new HashSet();
obs.add((PlFormula) parser.parseFormula("!f"));
var reviseeTweety = levi.revise(Kt, obs);
Console.WriteLine($"K revisee par !f = {reviseeTweety}");

var rtKb = new PlBeliefSet();
var itt = reviseeTweety.iterator();
while (itt.hasNext()) rtKb.add((PlFormula) itt.next());
var notF = (PlFormula) parser.parseFormula("!f");
Console.WriteLine($"Resultat incoherent ?         {reasoner.query(rtKb, faux)}");
Console.WriteLine($"Conclut desormais !f (ne vole pas) ? {rtKb.contains(notF)}");

// Quelle(s) croyance(s) la revision a-t-elle du sacrifier ?
var abandonnees = new java.util.ArrayList();
var itd = Kt.iterator();
while (itd.hasNext()) {
    var croyance = (PlFormula) itd.next();
    if (!reviseeTweety.contains(croyance)) abandonnees.add(croyance);
}
Console.WriteLine($"Croyance(s) abandonnee(s) par la revision : {abandonnees}");
Console.WriteLine();
Console.WriteLine("La revision a du abandonner une regle par defaut de la chaine p=>b=>f pour");
Console.WriteLine("integrer l'observation !f sans contradiction : c'est le coeur de l'AGM.");

K conclut que Tweety vole (f) ? true


K revisee par !f = [(b=>f), p, !f]


Resultat incoherent ?         false


Conclut desormais !f (ne vole pas) ? True


Croyance(s) abandonnee(s) par la revision : [(p=>b)]


La revision a du abandonner une regle par defaut de la chaine p=>b=>f pour


integrer l'observation !f sans contradiction : c'est le coeur de l'AGM.


## Conclusion

Ce notebook a execute, en C# natif via IKVM, les operateurs AGM de Tweety :

- **test de coherence** par entailment d'une contradiction (`SimplePlReasoner`) : une base
  est incoherente ssi elle entraine `a && !a` ;
- **contraction par noyaux** (`KernelContractionOperator` + `RandomIncisionFunction`) ;
- **revision** par l'**identite de Levi** (`LeviMultipleBaseRevisionOperator`) ;
- une application non-monotone (le pingouin Tweety) ou la revision resout une incoherence
  **derivee** en abandonnant une regle par defaut.

La revision de croyances est au coeur des systemes intelligents qui doivent **apprendre en
presence d'informations contradictoires** : mise a jour de bases de connaissances, fusion de
sources, diagnostic. Le notebook Python compagnon poursuit avec les **mesures d'incoherence**
et l'enumeration de **MUS** (sous-ensembles incoherents minimaux).

## Exercices

Les exercices ci-dessous reutilisent les operateurs construits plus haut (`levi`, la contraction
par noyaux, le raisonneur `reasoner`, le `parser`). Completez les stubs ; la base reste executable
de bout en bout meme si les exercices ne sont pas remplis.

In [8]:
// Exercice 1 : reviser votre propre base de croyances.
// Objectif : construire K = { pluie, pluie => sol_mouille } puis reviser par !sol_mouille.
// Indice : reutilisez le patron levi.revise(base, nouvelleInfo) et testez la coherence.
// Etape 1 : construire la HashSet baseEx1 avec les deux formules.
// Etape 2 : construire nouvelleInfoEx1 = { !sol_mouille }.
// Etape 3 : appeler levi.revise(...) et afficher le resultat + sa coherence.

// var baseEx1 = new HashSet();
// TODO : completer
Console.WriteLine("Exercice 1 a completer");

Exercice 1 a completer


In [9]:
// Exercice 2 : contraction par noyaux.
// Objectif : partir de K = { x, x=>y, y=>z } et la contracter par z.
// Indice : new KernelContractionOperator(new RandomIncisionFunction(), new SimplePlReasoner()).
// Etape 1 : construire la base et l'ensemble { z } a retirer.
// Etape 2 : appeler .contract(base, aRetirer).
// Etape 3 : verifier avec un SimplePlReasoner que la base contractee n'implique plus z.

// TODO : completer
Console.WriteLine("Exercice 2 a completer");

Exercice 2 a completer


In [10]:
// Exercice 3 : revision vs union naive (mesure de l'ecart).
// Objectif : sur K = { m, m=>n, !o }, comparer |K + {!n}| (union naive, incoherente)
//            au cardinal de la base revisee levi.revise(K, {!n}) (coherente).
// Indice : la revision retire au moins une formule ; comptez .size() de chaque cote.
// Etape 1 : construire K et la nouvelle information { !n }.
// Etape 2 : calculer l'union naive et la base revisee.
// Etape 3 : afficher les deux cardinaux et la difference (nombre de croyances sacrifiees).

// TODO : completer
Console.WriteLine("Exercice 3 a completer");

Exercice 3 a completer
